# University Waste Management & Agricultural Field Monitoring System
### HSV Green-Masking + YOLOv8 — Colab GPU Pipeline

This notebook runs the full pipeline end-to-end on a Colab GPU runtime:

1. Environment setup (GPU check, Drive mount, dependencies)
2. Dataset download (Kaggle) + sanity inspection
3. Dataset assembly: single-class merge, train/val/test split
4. HSV green-masking preprocessing (visual pipeline check)
5. Baseline (unmasked) vs HSV-masked training — the core ablation
6. Metrics comparison (mAP50, mAP50-95, precision, recall)
7. Inference on original RGB images with green-ratio post-filter
8. Persist everything to Google Drive

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better).

## 1. Environment setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/university-waste-management'
import os
os.makedirs(PROJECT_ROOT, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Clone the project repo (containing `src/`) into the Colab runtime. Replace `REPO_URL`
if you've pushed this project to GitHub — otherwise upload the `src/` folder manually via the
Colab file browser into `/content/University-Waste-Management/`.

In [ ]:
REPO_URL = ''  # e.g. 'https://github.com/<you>/university-waste-management.git'
CODE_DIR = '/content/University-Waste-Management'

if REPO_URL:
    !git clone -q {REPO_URL} {CODE_DIR}
else:
    os.makedirs(CODE_DIR, exist_ok=True)
    print('REPO_URL not set — upload/sync src/ into', CODE_DIR, 'manually before continuing.')

import sys
sys.path.insert(0, CODE_DIR)

In [ ]:
!pip install -q ultralytics opencv-python-headless kaggle pyyaml tqdm seaborn

## 2. Dataset download (Kaggle)

Upload your `kaggle.json` API token when prompted (Kaggle account → Settings → Create New API Token).

In [ ]:
from google.colab import files
import os

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
    uploaded = files.upload()  # select kaggle.json
    for fname in uploaded:
        os.rename(fname, f'{kaggle_dir}/kaggle.json')
    os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Kaggle credentials ready.')

In [ ]:
RAW_DATA_DIR = f'{PROJECT_ROOT}/data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)

!kaggle datasets download -d ravirajsinh45/crop-and-weed-detection-data-with-bounding-boxes -p {RAW_DATA_DIR} --unzip
!find {RAW_DATA_DIR} -maxdepth 3 | head -30

## 3. Dataset assembly

Discover image/label pairs, **inspect the class distribution before trusting the 0/1 class-id convention**, merge crop+weed into a single `green_vegetation` class, and split train/val/test.

In [ ]:
from pathlib import Path
from src.preprocessing.dataset_prep import (
    discover_pairs, inspect_class_distribution, split_dataset, write_data_yaml
)

pairs = discover_pairs(Path(RAW_DATA_DIR))
print(f'Found {len(pairs)} image/label pairs')
print('Class distribution (verify before merging!):', inspect_class_distribution(pairs))

In [ ]:
BASELINE_DIR = Path(PROJECT_ROOT) / 'data' / 'baseline'
split_dataset(pairs, BASELINE_DIR, train=0.7, val=0.2, test=0.1, seed=42, merge_to_single_class=True)

baseline_yaml = write_data_yaml(BASELINE_DIR / 'data.yaml', BASELINE_DIR, names=['green_vegetation'])
print('Baseline data.yaml:', baseline_yaml)

## 4. HSV green-masking — visual sanity check

Inspect the mask and both masking strategies (hard black-out vs soft desaturation) on a few sample images before committing to a full-dataset pass.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.preprocessing.hsv_mask import MaskConfig, visualize_pipeline

sample_paths = list((BASELINE_DIR / 'images' / 'train').iterdir())[:3]
mask_config = MaskConfig()  # tune DEFAULT_LOWER_GREEN/UPPER_GREEN in hsv_mask.py if needed

fig, axes = plt.subplots(len(sample_paths), 4, figsize=(16, 4 * len(sample_paths)))
for row, img_path in enumerate(sample_paths):
    image = cv2.imread(str(img_path))
    mask, hard, soft = visualize_pipeline(image, mask_config)
    for col, (title, img) in enumerate([
        ('original', image), ('mask', mask), ('hard-masked', hard), ('soft-masked', soft)
    ]):
        ax = axes[row, col] if len(sample_paths) > 1 else axes[col]
        cmap = 'gray' if img.ndim == 2 else None
        disp = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(disp, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
fig.tight_layout()
plt.show()

**If the mask misses vegetation or catches soil/straw**, tune `DEFAULT_LOWER_GREEN` /
`DEFAULT_UPPER_GREEN` in `src/preprocessing/hsv_mask.py` (or pass a custom `MaskConfig` above)
and re-run this cell before generating the full masked dataset.

In [ ]:
from src.preprocessing.dataset_prep import build_masked_variant

MASKED_DIR = Path(PROJECT_ROOT) / 'data' / 'masked'
build_masked_variant(BASELINE_DIR, MASKED_DIR, mode='soft', config=mask_config)
masked_yaml = write_data_yaml(MASKED_DIR / 'data.yaml', MASKED_DIR, names=['green_vegetation'])
print('Masked data.yaml:', masked_yaml)

## 5. Train — baseline vs HSV-masked ablation

Same architecture, same hyperparameters, only the preprocessing differs. This is the core evidence for the project's thesis.

In [ ]:
from src.training.train import TrainConfig, train

RUNS_DIR = f'{PROJECT_ROOT}/runs'

baseline_cfg = TrainConfig(
    data_yaml=str(baseline_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='baseline',
)
baseline_model, baseline_results = train(baseline_cfg)

In [ ]:
masked_cfg = TrainConfig(
    data_yaml=str(masked_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='masked',
)
masked_model, masked_results = train(masked_cfg)

## 6. Compare metrics

In [ ]:
from src.evaluation.metrics import load_results_csv, plot_loss_curves, plot_map_curves, compare_runs

baseline_run_dir = f'{RUNS_DIR}/baseline'
masked_run_dir = f'{RUNS_DIR}/masked'

baseline_df = load_results_csv(baseline_run_dir)
masked_df = load_results_csv(masked_run_dir)

plot_loss_curves(baseline_df, f'{PROJECT_ROOT}/reports/baseline_loss.png', 'Baseline — Loss')
plot_loss_curves(masked_df, f'{PROJECT_ROOT}/reports/masked_loss.png', 'HSV-Masked — Loss')
plot_map_curves(baseline_df, f'{PROJECT_ROOT}/reports/baseline_map.png', 'Baseline — mAP')
plot_map_curves(masked_df, f'{PROJECT_ROOT}/reports/masked_map.png', 'HSV-Masked — mAP')

summary = compare_runs(baseline_run_dir, masked_run_dir, f'{PROJECT_ROOT}/reports/comparison.png')
summary

## 7. Inference on original RGB images

The masked model still runs on **unmasked** images at inference time — masking is a training-time noise filter only. The optional green-ratio post-filter drops boxes that don't actually contain green pixels.

In [ ]:
from src.inference.predict import predict_and_save

test_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]
masked_weights = f'{masked_run_dir}/weights/best.pt'

for img_path in test_images:
    out_path = f'{PROJECT_ROOT}/reports/inference/{img_path.stem}_pred.jpg'
    predict_and_save(
        masked_weights, img_path, out_path,
        conf=0.25, green_ratio_threshold=0.15, mask_config=mask_config,
    )
print('Saved annotated predictions to', f'{PROJECT_ROOT}/reports/inference/')

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

preds = sorted(Path(f'{PROJECT_ROOT}/reports/inference').glob('*_pred.jpg'))[:5]
fig, axes = plt.subplots(1, len(preds), figsize=(4 * len(preds), 4))
for ax, p in zip(axes, preds):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(p.stem)
    ax.axis('off')
fig.tight_layout()
plt.show()

## 8. Everything is already on Drive

Since `PROJECT_ROOT` lives under `/content/drive/MyDrive/...`, weights (`runs/*/weights/best.pt`), plots (`reports/`), and the assembled datasets all persist automatically across Colab sessions — no extra export step needed.

In [ ]:
print('Baseline weights:', f'{baseline_run_dir}/weights/best.pt')
print('Masked weights:  ', f'{masked_run_dir}/weights/best.pt')
print('Reports:         ', f'{PROJECT_ROOT}/reports/')
print()
print('Final metrics summary:')
summary